# Minimal reproducer: `vmap` + `torch.compile(backend='inductor')` correctness bug

Shows that for Z2-symmetric fermionic PEPS with **odd block dimensions >= 5** (D=10,14,18,...),
`torch.compile` after `torch.export` + `torch.vmap` produces **wrong results**.

Even block dims and small odd block dims (D<=6) are unaffected.

**Setup:** exact contraction (no SVD/QR truncation), so the bug is purely in inductor kernel generation.

In [13]:
import numpy as np
import quimb as qu
import quimb.tensor as qtn
import symmray as sr
import torch
from vmc_torch.GPU.vmc_utils import random_initial_config

torch.set_default_device("cuda")
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

In [14]:
Lx, Ly = 6, 4
nsites = Lx * Ly
B = 2  # batch size

## Define amplitude function and model wrapper

In [15]:
def amplitude(x, tn):
    """Single-sample exact contraction of fermionic PEPS."""
    tnx = tn.isel(
        {tn.site_ind(site): x[i] for i, site in enumerate(tn.sites)}
    )
    # exact contraction — no SVD/QR truncation
    return tnx.contract()

In [16]:
class TNAmplitudeModel(torch.nn.Module):
    """Wraps a TN amplitude function as an nn.Module for export+compile."""

    def __init__(self, fn, tn):
        super().__init__()
        params, skeleton = qtn.pack(tn)
        params_flat, params_pytree = qu.utils.tree_flatten(
            params, get_ref=True
        )
        self.params = torch.nn.ParameterList(
            [torch.as_tensor(x, dtype=torch.float64) for x in params_flat]
        )
        self._skeleton = skeleton
        self._params_pytree = params_pytree
        self._fn = fn

    def _amplitude(self, x, *flat_params):
        params = qu.utils.tree_unflatten(list(flat_params), self._params_pytree)
        tn = qtn.unpack(params, self._skeleton)
        return self._fn(x, tn)

    def forward(self, x):
        return self._amplitude(x, *self.params)

## Generate random configs

In [38]:
N_f = nsites-2  # number of fermions (half-filling-ish)

# rng = np.random.default_rng(0)
# xs_np = np.zeros((B, nsites), dtype=np.int64)
# for i in range(B):
#     occ = np.zeros(nsites, dtype=np.int64)
#     occ[:N_f] = 1
#     rng.shuffle(occ)
#     xs_np[i] = occ
# xs = torch.tensor(xs_np, device="cuda")

z2_map = {0:0, 1:2, 2:3, 3:1}
xs = [
    random_initial_config(N_f, nsites, seed=s)
    for s in range(B)
]
for x in xs:
    x.copy_(torch.tensor([z2_map[int(xi)] for xi in x], device="cuda"))
xs = torch.stack(xs)
print(f"Configs shape: {xs.shape}")
xs

Configs shape: torch.Size([2, 24])


tensor([[3, 3, 3, 3, 3, 0, 3, 3, 2, 3, 2, 2, 2, 2, 2, 3, 3, 3, 2, 0, 2, 2, 2, 2],
        [0, 2, 2, 3, 3, 0, 3, 2, 2, 2, 3, 3, 2, 2, 2, 3, 3, 2, 3, 3, 2, 3, 3, 2]],
       device='cuda:0')

## Test function: compare eager vs export+compile

In [39]:
def test_D(D):
    """Test eager vs export+vmap+compile for a given bond dim D."""
    print(f"\n{'=' * 60}")
    print(f"D={D}, Z2 block dim={D // 2}, exact contraction, {Lx}x{Ly}")
    print(f"{'=' * 60}")

    # random Z2-symmetric fermionic PEPS
    peps = sr.networks.PEPS_fermionic_rand(
        "Z2", Lx, Ly, D,
        # phys_dim=4,
        phys_dim=[
                (0, 0),
                (1, 0),
                (1, 1),
                (0, 1),
        ],
        subsizes="equal",
        flat=True,
        seed=42,
        dtype="float64",
    )
    for ts in peps.tensors:
        ts_data = ts.data
        ts_data.indices[-1]._linearmap = None
        ts.modify(data=ts_data)

    # --- Eager baseline ---
    model = TNAmplitudeModel(amplitude, peps)
    params_list = list(model.params)

    # --- torch.export ---
    from torch.export import export

    class _AmpModule(torch.nn.Module):
        def __init__(self, amp_fn):
            super().__init__()
            self._fn = amp_fn

        def forward(self, x, *flat_params):
            return self._fn(x, *flat_params)

    with torch.no_grad():
        exported = export(
            _AmpModule(model._amplitude),
            (xs[0], *params_list),
        )
    exported_module = exported.module()

    # --- torch.vmap ---
    n_params = len(params_list)
    vmapped = torch.vmap(
        exported_module,
        in_dims=(0, *([None] * n_params)),
    )

    with torch.inference_mode():
        vmap_eager = vmapped(xs, *params_list)
        print(f"  Vmap eager:  {vmap_eager}")

    # --- torch.compile ---
    torch._dynamo.reset()
    compiled = torch.compile(vmapped, backend="inductor")

    with torch.inference_mode():
        compiled_results = compiled(xs, *params_list)
        torch.cuda.synchronize()

    diff = (vmap_eager - compiled_results).abs().max().item() / vmap_eager.abs().max().item()
    ok = diff < 1e-6
    print(f"  Compiled vs eager:        {diff:.2e}  {'PASS' if ok else 'FAIL'}")
    print(f"    eager:    {vmap_eager}")
    print(f"    compiled: {compiled_results}")

    return ok, diff

## Run tests across bond dimensions

In [40]:
D_list = [10]#  8, 10
results = {}
for D in D_list:
    results[D] = test_D(D)


D=10, Z2 block dim=5, exact contraction, 6x4
  Vmap eager:  tensor([-1.2778e+15, -5.5105e+15], device='cuda:0')
  Compiled vs eager:        4.99e-16  PASS
    eager:    tensor([-1.2778e+15, -5.5105e+15], device='cuda:0')
    compiled: tensor([-1.2778e+15, -5.5105e+15], device='cuda:0')


In [ ]:
print(f"\n{'=' * 60}")
print("SUMMARY")
print(f"{'=' * 60}")
print(f"  {'D':>4}  {'Z2 block dim':>12}  {'status':>8}  {'max diff':>10}")
print(f"  {'-'*4}  {'-'*12}  {'-'*8}  {'-'*10}")
for D in D_list:
    ok, diff = results[D]
    bdim = D // 2
    print(
        f"  {D:>4}  {bdim:>12}  "
        f"{'PASS' if ok else 'FAIL':>8}  {diff:>10.2e}"
    )
print(f"{'=' * 60}")

## Expected result

| D | Z2 block dim | Status | Notes |
|---|---|---|---|
| 4 | 2 | PASS | |
| 6 | 3 | PASS | |
| 8 | 4 | PASS | |
| **10** | **5** | **FAIL** | odd block dim |
| 12 | 6 | PASS | |
| **14** | **7** | **FAIL** | odd block dim |
| 16 | 8 | PASS | |
| **18** | **9** | **FAIL** | odd block dim |
| 20 | 10 | PASS | |

**Pattern:** Bug hits odd Z2 block dims >= 5. Even block dims and small odd (2, 3) are fine.

Eager vmap is always correct — the bug is purely in the inductor backend.